# 🚘 License Plate Detection & Recognition — Google Colab

This notebook runs the full pipeline (YOLOv8 detection + EasyOCR recognition) on a free Colab GPU:
1. Get the project onto Colab (clone from GitHub, or upload the zip)
2. Install dependencies
3. (Optional) Mount Google Drive so trained weights survive after the session ends
4. Get a labeled dataset
5. Train YOLOv8
6. Run inference on an image / video
7. Download your results

**Before you start:** in the Colab menu, go to `Runtime → Change runtime type → Hardware accelerator → GPU` (T4 is fine).

## 1. Check GPU is enabled

In [ ]:
!nvidia-smi

## 2. Get the project onto Colab

**Option A — clone from your GitHub repo** (recommended, once you've pushed it):

In [ ]:
# Replace with your repo URL after you've pushed it (see the earlier README section
# "Deploying to GitHub"). Skip this cell and use Option B below if you haven't pushed yet.

# !git clone https://github.com/<your-username>/license-plate-detection.git
# %cd license-plate-detection

**Option B — upload the project zip directly** (works right now, no GitHub needed):

Run the cell below, then click "Choose Files" and select `license-plate-detection.zip`.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # select license-plate-detection.zip

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('license-plate-detection')

%cd license-plate-detection
!ls

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. (Optional) Mount Google Drive

Colab sessions are temporary — anything not saved to Drive disappears when the runtime resets.
Mounting Drive lets you save trained weights (`models/best.pt`) and outputs permanently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Folder in your Drive where we'll back up weights/outputs after training
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/license-plate-detection-backup'
import os
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

## 5. Get a labeled dataset

**Option A — Roboflow** (recommended — many ready-made, YOLOv8-format plate datasets):

In [ ]:
import os

# Get a free API key at https://roboflow.com (Settings → API Keys) and paste it below,
# or use Colab's "Secrets" (key icon in the left sidebar) to store it more safely.
os.environ['ROBOFLOW_API_KEY'] = 'YOUR_ROBOFLOW_API_KEY'

!python scripts/download_dataset.py --workspace <workspace> --project <project> --version 1

**Option B — upload your own dataset zip** (already in YOLO format: `images/train`, `images/val`, `labels/train`, `labels/val`):

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()  # select your dataset zip
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('data')  # should populate data/images/... and data/labels/...

!find data -maxdepth 3 -type d

## 6. Train YOLOv8

Colab's free GPU (usually a T4) trains far faster than CPU. 100 epochs on a few thousand
images typically takes well under an hour on a T4 — adjust `--epochs`/`--batch` for your
dataset size and Colab's session time limits.

In [ ]:
!python src/train.py --epochs 100 --batch 16 --model yolov8n.pt --device 0

**Back up the trained weights to Drive** (skip if you didn't mount Drive in step 4):

In [ ]:
import shutil
shutil.copy('models/best.pt', f'{DRIVE_BACKUP_DIR}/best.pt')
print(f"Saved to {DRIVE_BACKUP_DIR}/best.pt")

## 7. Run inference

Upload a test image or video, then run detection + OCR on it.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select an image or video to test
test_file = list(uploaded.keys())[0]

In [ ]:
!python src/detect.py --source "{test_file}" --no-display

**Display the annotated result inline:**

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

stem = Path(test_file).stem
out_path = f"outputs/{stem}_annotated.jpg"  # for images; videos save as _annotated.mp4 instead

if Path(out_path).exists():
    img = cv2.cvtColor(cv2.imread(out_path), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print(f"Look for your annotated output in the outputs/ folder (video files won't preview inline here).")

**View the logged plate readings:**

In [ ]:
import pandas as pd
pd.read_csv('outputs/results.csv')

## 8. Download your results

Grab the annotated output, the CSV log, and the trained weights back to your machine.

In [ ]:
from google.colab import files
import shutil

shutil.make_archive('lp_results', 'zip', 'outputs')
files.download('lp_results.zip')

# Also grab the trained weights directly:
# files.download('models/best.pt')

## 9. (Optional) Try the Streamlit demo via a public tunnel

Colab can't serve Streamlit on `localhost` for you to open directly, but you can tunnel it
with `pyngrok` to get a temporary public URL.

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok

# Free at https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")

get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

public_url = ngrok.connect(8501)
print(f"Open the app here: {public_url}")

---
### Notes on running this project in Colab

- **GPU vs CPU:** confirm step 1 shows a GPU — training on CPU in Colab is very slow and may exceed session limits.
- **Session limits:** free Colab disconnects idle sessions and has a maximum runtime (~12h). For long training runs, checkpoint to Drive periodically (Ultralytics does this automatically under `runs/train/.../weights/last.pt`) so you can resume with `python src/train.py --resume`.
- **EasyOCR on first use** downloads its recognition model (~100MB) — this happens automatically the first time `src/detect.py` or `app.py` runs OCR.
- **Persisting between sessions:** anything outside `/content/drive` is wiped when the Colab runtime recycles. Keep step 4's Drive mount active if you want `models/best.pt` and `outputs/` to survive.
